# 4. Explaining the model

**Question:** why does the model give a patient a high or low risk?

For logistic regression each feature adds a fixed amount to the prediction:

`contribution = coefficient × (patient value − average value)`

- positive → pushes the risk **up**
- negative → pushes the risk **down**

For a linear model this is exactly what the SHAP method computes. The function is in `src/explain.py`, and the app uses the same function.

In [ ]:
import sys
sys.path.append("..")

import pandas as pd
import joblib
import matplotlib.pyplot as plt
from src.explain import explain, FEATURES, LABELS

bundle = joblib.load("../models/model.joblib")
data = pd.read_csv("../data/clean.csv")

## Global view: what the model relies on
The coefficient of each prepared column. Bigger (positive or negative) = more influence.

In [ ]:
coef = bundle["model"][-1].coef_[0]
names = [n.split("__")[1] for n in bundle["column_names"]]

order = abs(coef).argsort()
plt.figure(figsize=(7, 6))
plt.barh([names[i] for i in order], coef[order])
plt.xlabel("Coefficient (positive = higher risk)")
plt.title("What the model relies on")
plt.tight_layout()
plt.show()

## One patient: why this risk?

In [ ]:
def show_patient(index):
    patient = data.loc[index, FEATURES].to_dict()
    risk, table = explain(bundle, patient)

    print("Real diagnosis:", "sick" if data.loc[index, "target"] == 1 else "healthy")
    print("Predicted risk:", round(risk, 2))

    top = table.head(6).iloc[::-1]
    colors = ["tab:red" if c > 0 else "tab:blue" for c in top["contribution"]]
    plt.figure(figsize=(6, 3.5))
    plt.barh(top["label"], top["contribution"], color=colors)
    plt.axvline(0, color="black", linewidth=0.8)
    plt.xlabel("Contribution (red = raises risk, blue = lowers risk)")
    plt.tight_layout()
    plt.show()

In [ ]:
# A sick patient
show_patient(data[data["target"] == 1].index[0])

In [ ]:
# A healthy patient
show_patient(data[data["target"] == 0].index[0])